# Auto Loc

In this notebook, the functions generate an image of each Id and also highlights there location on certain distribution plots.

## Import Libraries and Load Data

In [ ]:
#import libraries and set up tap service
import matplotlib.pyplot as plt
import numpy as np
from astropy.table import Table
import pyvo
import astropy.units as u
from lsst.rsp import get_tap_service
import pandas as pd
pd.set_option('display.max_columns', None)

from lsst.utils.plotting import (get_multiband_plot_colors,
                                 get_multiband_plot_symbols)

from lsst.daf.butler import Butler
from lsst.utils.plotting import (
    get_multiband_plot_colors,
    get_multiband_plot_symbols,
    get_multiband_plot_linestyles
)

service = get_tap_service("tap")
assert service is not None

import seaborn as sns
my_pal = sns.color_palette("colorblind")

import warnings
warnings.filterwarnings('error')

import lsst.afw.display as afwDisplay
import lsst.geom as geom
from astropy.wcs import WCS
import numpy as np
import gc

In [ ]:
Ex_diaObj_df = pd.read_csv("data/Extra_diaObj_df.csv",index_col=0)
forcedDiaObj_df = pd.read_csv("data/diaForced_df.csv",index_col=0)

grouped_forcedDia = forcedDiaObj_df.groupby("diaObjectId")

filter_names = ['u', 'g', 'r', 'i', 'z', 'y']
filter_colors = get_multiband_plot_colors()
filter_symbols = get_multiband_plot_symbols()
filter_linestyles = get_multiband_plot_linestyles()

In [ ]:
afwDisplay.setDefaultBackend("matplotlib")

butler = Butler("dp1", collections="LSSTComCam/DP1")
assert butler is not None

In [ ]:
#adding in center point of fields from 301_0_DP1
field_centers = {
    "47 Tuc": [6.02, -72.08],
    "SV 38 7": [37.86, 6.98],
    "Fornax": [40.00, -34.45],
    "ECDFS": [53.13, -28.10],
    "EDFS": [59.10, -48.73],
    "SV 95 -25": [95.00, -25.00],
    "Seagull": [106.23, -10.51],
}
field_names = list(field_centers.keys())

In [ ]:
#load in the maglim for each object
u_maglim = butler.get('deepCoadd_psf_maglim_consolidated_map_weighted_mean',
                      band='u')
g_maglim = butler.get('deepCoadd_psf_maglim_consolidated_map_weighted_mean',
                      band='g')
r_maglim = butler.get('deepCoadd_psf_maglim_consolidated_map_weighted_mean',
                      band='r')
i_maglim = butler.get('deepCoadd_psf_maglim_consolidated_map_weighted_mean',
                      band='i')
z_maglim = butler.get('deepCoadd_psf_maglim_consolidated_map_weighted_mean',
                      band='z')
y_maglim = butler.get('deepCoadd_psf_maglim_consolidated_map_weighted_mean',
                      band="y")

## Defining Image Functions

In [ ]:
def remove_figure(fig):
    """
    Remove a figure to reduce memory footprint.

    Parameters
    ----------
    fig: matplotlib.figure.Figure
        Figure to be removed.

    Returns
    -------
    None
    """

    for ax in fig.get_axes():
        for im in ax.get_images():
            im.remove()

    fig.clf()
    plt.close(fig)
    gc.collect()

In [ ]:
def zoom_in(ax, wcs, ra, dec, side):
    """
    Zoom in on the search location.

    Parameters
    ----------
    ax: matplotlib.axes._axes.Axes
        Axes for showing a plot.
    wcs: lsst.afw.geom.SkyWcs
        WCS of an image.
    ra: float
        RA of a celestial location in degree.
    dec: float
        DEC of a celestial location in degree.
    side: float
        Size of the display region in arcseconds.

    Returns
    -------
    None
    """

    x, y = wcs.skyToPixel(geom.SpherePoint(ra, dec, geom.degrees))
    pixel_scale = 0.2
    side_pix = side/pixel_scale
    ax.set_xlim([x - side_pix/2, x + side_pix/2])
    ax.set_ylim([y - side_pix/2, y + side_pix/2])

In [ ]:
def make_plot(deep_coadd,row,tx,tx2,objects,objects2,directory=""):

    diaObjectId = row["diaObjectId"]
    ra_val = row["ra"]
    dec_val = row["dec"]

    deep_coadd_extent = (deep_coadd.getBBox().beginX,
                     deep_coadd.getBBox().endX,
                     deep_coadd.getBBox().beginY,
                     deep_coadd.getBBox().endY)

    fig = plt.figure(figsize=(10, 10))
    ax1 = fig.add_subplot(121, projection=WCS(deep_coadd.getWcs().getFitsMetadata()))
    
    
    im = ax1.imshow(deep_coadd.image.array, cmap='gray',
                    vmin=-75, vmax=125,
                    extent=deep_coadd_extent,
                    origin='lower')
    
    ax1.scatter(objects[tx]['r_centroid_x'], objects[tx]['r_centroid_y'],
                facecolors='none', edgecolors='orange',label="Low SNR Obj")

    ax1.scatter(objects2[tx2]['r_centroid_x'], objects2[tx2]['r_centroid_y'],
                facecolors='none', edgecolors='red',label="High SNR Obj")
    
    
    # Calculate the center pixel
    center_x = (deep_coadd_extent[1] + deep_coadd_extent[0]) / 2
    center_y = (deep_coadd_extent[-1] + deep_coadd_extent[-2]) / 2
    
    # Plot it (using the same coordinate system as your objects)
    # ax1.scatter(center_x, center_y, color='cyan', marker='x', s=100, label='Pixel Center')

    ax1.scatter(ra_val, dec_val, 
            transform=ax1.get_transform('world'), 
            marker='+', 
            s=300, 
            color='lime', 
            label=f'diaObj RA/Dec:\n{ra_val:.2f}/{dec_val:.2f}')
    
    # Access the RA and Dec coordinate helpers from the axes
    ra_scale = ax1.coords[0]
    dec_scale = ax1.coords[1]

    
    # Set the format to decimal degrees ('.degree' or 'd.dd')
    ra_scale.set_major_formatter('d.dd')
    dec_scale.set_major_formatter('d.dd')
    
    # Optional: Adjust the number of ticks if it gets crowded
    ra_scale.set_ticks(number=5)
    dec_scale.set_ticks(number=5)
    
    ax1.set_xlabel('Right Ascension',fontsize=14)
    ax1.set_ylabel('Declination',fontsize=14)
    
    for ax in [ax1]:
        ax.grid(color='white', ls='solid')

    side = 140

    zoom_in(ax1, deep_coadd.getWcs(), ra_val, dec_val, side)

    plt.title(f"diaObjectId: {diaObjectId}\n(In_Obj: {row["in_Obj"]})",fontsize=16)
    plt.legend(loc="center left", bbox_to_anchor=(1.01, 0.5))
    plt.tight_layout()
    if directory:
        plt.savefig(f"{directory}/coadd_image.png",dpi=330,bbox_inches="tight")
    else:
        plt.show()
        pass
    remove_figure(fig)

In [ ]:
def make_field_plot(row,path=""):
    name = row["DR3_field_loc_center"]
    ra_cen, dec_cen = field_centers[name]
    span_dec = 0.75
    if name == 'SV 38 7':
        span_dec = 1.00
    span_ra = span_dec / np.cos(np.deg2rad(dec_cen))
    ra = np.linspace(ra_cen-span_ra, ra_cen+span_ra, 250)
    dec = np.linspace(dec_cen-span_dec, dec_cen+span_dec, 250)

    x, y = np.meshgrid(ra, dec)
    values = r_maglim.get_values_pos(x, y)
    row1, col = np.where(values < 0.0)
    values[row1, col] = 'NaN'
    plt.pcolormesh(x, y, values, vmin=24.0, vmax=27)
    plt.colorbar(label="PSF Mag Limit (r-band)")

    size = 20

    conversion_mas_deg = 1 / 3600000

    current_stars = Ex_diaObj_df[Ex_diaObj_df["DR3_field_loc_center"] == name]
    plt.errorbar(current_stars["DR3_ra"],current_stars["DR3_dec"],current_stars["DR3_ra_error"] * conversion_mas_deg,
                 current_stars["DR3_dec_error"] * conversion_mas_deg,fmt="none",elinewidth=size/2,c="k")
    plt.scatter(current_stars["DR3_ra"],current_stars["DR3_dec"],s=size,c="k",label=name)
    plt.scatter(row["ra"],row["dec"],s=size*2,c="y",marker="*",label="Current Star")

    plt.xlabel("Right Ascension (deg)",fontsize=16)
    plt.ylabel("Declination (deg)",fontsize=16)
    plt.title(f"diaObjectId: {row["diaObjectId"]}",fontsize=20)
    plt.legend()
    plt.gca().invert_xaxis()
    plt.tight_layout()
    if path:
        plt.savefig(f"{path}/{name}_field_plot.png",dpi=330,bbox_inches="tight")
    else:
        plt.show()
    plt.close()

In [ ]:
def mag_mag_spotlight(row,path=""):
    #pull mask defined in dia_stats
    mask = Ex_diaObj_df["comb_auto_filter"]
    
    #plot mag in Rubin vs GAIA to check for outliers
    fig, ax = plt.subplots()
    
    p = sns.scatterplot(Ex_diaObj_df[mask], x = "DR3_phot_g_mean_mag", y = "filt_mean_r_mag",
                style = "in_Obj",style_order=[True,False],
                legend=True,hue="nr",palette = "vlag",edgecolor="k")
    
    sns.lineplot(x = np.linspace(np.min(Ex_diaObj_df[mask]["filt_mean_r_mag"]), np.max(Ex_diaObj_df[mask]["filt_mean_r_mag"]),1000),
             y = np.linspace(np.min(Ex_diaObj_df[mask]["filt_mean_r_mag"]), np.max(Ex_diaObj_df[mask]["filt_mean_r_mag"]),1000),
             c = "k",alpha=.5,linestyle=":",label="1:1 Line",legend=True)

    plt.scatter(row["DR3_phot_g_mean_mag"],row["filt_mean_r_mag"],c="y",s=50,marker="*",label="Current Star")

    
    handles, labels = p.get_legend_handles_labels()
    
    norm = plt.Normalize(Ex_diaObj_df["nr"].min(), Ex_diaObj_df["nr"].max())
    sm = plt.cm.ScalarMappable(cmap="vlag", norm=norm)
    sm.set_array([])
    
    plt.xlabel("Average G Mag from GAIA",fontsize=16)
    plt.ylabel("Average R Mag from Rubin",fontsize=16)
    plt.title("R Mag in Rubin vs G Mag in GAIA",fontsize=20)
    plt.legend(handles=handles[-4:], labels=labels[-4:])
    plt.gca().invert_yaxis()
    plt.gca().invert_xaxis()
    
    cax = fig.add_axes([0.92, 0.15, 0.03, 0.7])
    p.figure.colorbar(sm,cax=cax)
    if path:
        plt.savefig(f"{path}/obj_mag_mag_comp.png",dpi=330,bbox_inches="tight")
    else:
        plt.show()
    plt.close()
    pass

In [ ]:
def epoch_mag_spotlight(row,path=""):
    #pull mask defined in dia_stats
    mask = Ex_diaObj_df["comb_auto_filter"]
    
    #plot it against mean mag in r
    fig, ax = plt.subplots()
    
    p = sns.scatterplot(Ex_diaObj_df[mask], y = "filt_mean_r_mag", x = "nSources",
                style = "in_Obj",style_order=[True,False],
                legend=True,hue="nr",palette = "vlag",edgecolor="k")
    
    handles, labels = p.get_legend_handles_labels()

    norm = plt.Normalize(Ex_diaObj_df["nr"].min(), Ex_diaObj_df["nr"].max())
    sm = plt.cm.ScalarMappable(cmap="vlag", norm=norm)
    sm.set_array([])
    
    plt.ylabel("Mean R Mag (From Rubin)",fontsize=16)
    plt.xlabel("Number of Epochs Observed",fontsize=16)
    plt.xscale("log")
    plt.title("Mean R Mag versus # of Epochs",fontsize=20)
    plt.legend(handles=handles[-3:], labels=labels[-3:])
    plt.gca().invert_yaxis()
    
    cax = fig.add_axes([0.92, 0.15, 0.03, 0.7])
    p.figure.colorbar(sm,cax=cax)
    if path:
        plt.savefig(f"{path}/obj_mag_epoch_comp.png",dpi=330,bbox_inches="tight")
    else:
        plt.show()
    plt.close()
    pass

In [ ]:
def pull_out_row(row,path=""):
    print(f"Making location plots for {row["diaObjectId"]}...")

    """
    Making Coadd Plot for Star
    """
    
    ra,dec = row[["ra","dec"]]

    band = "r"

    query = "band.name = :band AND patch.region OVERLAPS POINT(:ra, :dec)"
    bind = {'band': band, 'ra': ra, 'dec': dec}
    deep_coadd_refs = butler.query_datasets("deep_coadd", where=query,
                                            bind=bind, order_by='patch')

    # for ref in deep_coadd_refs:
    #     dataId = ref.dataId
    #     use_columns = ['objectId', 'patch',
    #                'r_centroid_x', 'r_centroid_y',
    #                'r_psfFlux', 'r_psfFluxErr',"coord_ra","coord_dec"]
    #     objects = butler.get('object', tract=dataId.get('tract'),
    #                          parameters={'columns': use_columns})
    
    #     if np.min(objects["coord_ra"]) < ra and ra < np.max(objects["coord_ra"]):
    #         if np.min(objects["coord_dec"]) < dec and dec < np.max(objects["coord_dec"]):
    #             deep_coadd = butler.get(ref)
    #             break

    
    for ref in deep_coadd_refs:
        dataId = ref.dataId
        use_columns = ['objectId', 'patch',
                   'r_centroid_x', 'r_centroid_y',
                   'r_psfFlux', 'r_psfFluxErr',"coord_ra","coord_dec"]
        objects = butler.get('object', tract=dataId.get('tract'),
                             parameters={'columns': use_columns})

        filt_objects = objects[objects['patch'] == dataId.get('patch')]

        deep_coadd = butler.get(ref)
        
        if np.min(filt_objects["coord_ra"]) < ra and ra < np.max(filt_objects["coord_ra"]):
            if np.min(filt_objects["coord_dec"]) < dec and dec < np.max(filt_objects["coord_dec"]):
                break


    deep_coadd_extent = (deep_coadd.getBBox().beginX,
                     deep_coadd.getBBox().endX,
                     deep_coadd.getBBox().beginY,
                     deep_coadd.getBBox().endY)



    use_columns = ['objectId', 'patch',
               'r_centroid_x', 'r_centroid_y',
               'r_psfFlux', 'r_psfFluxErr']
    
    objects = butler.get('object', tract=dataId.get('tract'),
                     parameters={'columns': use_columns})

    tx = np.where(objects['patch'] == dataId.get('patch'))[0]
    # print("Number of objects: ", len(tx))

    sel = objects['r_psfFlux'] / objects['r_psfFluxErr'] > 300
    objects2 = objects[sel]
    tx2 = np.where(objects2['patch'] == dataId.get('patch'))[0]
    # print("Number of high SNR objects: ", len(tx2))
    # print(f"Stellar ra/dec: {ra} {dec}")
    # print(f"in_Obj: {row["in_Obj"]}")

    if path:
        # Creating New Folder for current star
        dia_path = os.path.join(path,f"{row["diaObjectId"]}")
        
        if not os.path.isdir(dia_path):
            os.mkdir(dia_path)
    
    make_plot(deep_coadd,row,tx,tx2,objects,objects2,directory=dia_path)

    """
    Making Distribution Plots with Star Highlighted
    """
    
    make_field_plot(row,dia_path)

    mag_mag_spotlight(row,dia_path)

    epoch_mag_spotlight(row,dia_path)
    
    

In [ ]:
def main():

    outdir = os.getcwd()
    auto_path = os.path.join(outdir,"auto_plots")
    
    if not os.path.isdir(auto_path):
        os.mkdir(auto_path)

    #pull mask defined in dia_stats
    mask = Ex_diaObj_df["comb_auto_filter"]

    Ex_diaObj_df[mask].apply(pull_out_row, path=auto_path, axis=1)

In [ ]:
main()

## Additional Querying Features for Coadd Image

In [ ]:
#ensuring this DOES NOT run if all cells are ran by accident
assert True == False

In [ ]:
def query_make_plot(deep_coadd,row,tx,tx2,objects,objects2,display=True):

    outdir = os.getcwd()
    auto_path = os.path.join(outdir,"auto_plots")
    
    if not os.path.isdir(auto_path):
        os.mkdir(auto_path)

    diaObjectId = row["diaObjectId"]
    ra_val = row["ra"]
    dec_val = row["dec"]

    deep_coadd_extent = (deep_coadd.getBBox().beginX,
                     deep_coadd.getBBox().endX,
                     deep_coadd.getBBox().beginY,
                     deep_coadd.getBBox().endY)

    fig = plt.figure(figsize=(10, 10))
    ax1 = fig.add_subplot(121, projection=WCS(deep_coadd.getWcs().getFitsMetadata()))
    
    
    im = ax1.imshow(deep_coadd.image.array, cmap='gray',
                    vmin=-75, vmax=125,
                    extent=deep_coadd_extent,
                    origin='lower')
    
    ax1.scatter(objects[tx]['r_centroid_x'], objects[tx]['r_centroid_y'],
                facecolors='none', edgecolors='orange',label="Low SNR Obj")

    ax1.scatter(objects2[tx2]['r_centroid_x'], objects2[tx2]['r_centroid_y'],
                facecolors='none', edgecolors='red',label="High SNR Obj")
    
    
    # Calculate the center pixel
    center_x = (deep_coadd_extent[1] + deep_coadd_extent[0]) / 2
    center_y = (deep_coadd_extent[-1] + deep_coadd_extent[-2]) / 2
    
    # Plot it (using the same coordinate system as your objects)
    # ax1.scatter(center_x, center_y, color='cyan', marker='x', s=100, label='Pixel Center')

    ax1.scatter(ra_val, dec_val, 
            transform=ax1.get_transform('world'), 
            marker='+', 
            s=50, 
            color='lime', 
            label=f'diaObj RA/Dec:\n{ra_val:.2f}/{dec_val:.2f}')
    
    # Access the RA and Dec coordinate helpers from the axes
    ra_scale = ax1.coords[0]
    dec_scale = ax1.coords[1]

    
    # Set the format to decimal degrees ('.degree' or 'd.dd')
    ra_scale.set_major_formatter('d.dd')
    dec_scale.set_major_formatter('d.dd')
    
    # Optional: Adjust the number of ticks if it gets crowded
    ra_scale.set_ticks(number=5)
    dec_scale.set_ticks(number=5)
    
    ax1.set_xlabel('Right Ascension',fontsize=14)
    ax1.set_ylabel('Declination',fontsize=14)
    
    for ax in [ax1]:
        ax.grid(color='white', ls='solid')

    side = 140

    zoom_in(ax1, deep_coadd.getWcs(), ra_val, dec_val, side)

    plt.title(f"diaObjectId: {diaObjectId}\n(In_Obj: {row["in_Obj"]})",fontsize=16)
    plt.legend(loc="center left", bbox_to_anchor=(1.01, 0.5))
    plt.tight_layout()
    if display:
        plt.show()
    else:
        plt.savefig(f"{auto_path}/{diaObjectId}/coadd_image.png",dpi=330,bbox_inches="tight")
        plt.show()
        pass
    remove_figure(fig)

In [ ]:
def query_main():

    user_inp = input("Please enter the last four digits of diaObjectId: ")
    user_dis = input("Would you like to save the figure? (y/n) ")

    if user_dis.strip().lower() == "y":
        dis = False
    else:
        dis = True

    rows = Ex_diaObj_df[Ex_diaObj_df["diaObjectId"].astype(str).str.endswith(user_inp)]

    if len(rows) > 1:
        print("Multiple rows found, please re-run & add another digit")
        return None
    elif len(rows) == 0:
        print("Nothing found, please try again!")
        return None
    else:
        row = rows.iloc[0]

    ra,dec = row[["ra","dec"]]

    band = "r"

    query = "band.name = :band AND patch.region OVERLAPS POINT(:ra, :dec)"
    bind = {'band': band, 'ra': ra, 'dec': dec}
    deep_coadd_refs = butler.query_datasets("deep_coadd", where=query,
                                            bind=bind, order_by='patch')


    for ref in deep_coadd_refs:
        dataId = ref.dataId
        use_columns = ['objectId', 'patch',
                   'r_centroid_x', 'r_centroid_y',
                   'r_psfFlux', 'r_psfFluxErr',"coord_ra","coord_dec"]
        objects = butler.get('object', tract=dataId.get('tract'),
                             parameters={'columns': use_columns})

        filt_objects = objects[objects['patch'] == dataId.get('patch')]

        deep_coadd = butler.get(ref)
        
        if np.min(filt_objects["coord_ra"]) < ra and ra < np.max(filt_objects["coord_ra"]):
            if np.min(filt_objects["coord_dec"]) < dec and dec < np.max(filt_objects["coord_dec"]):
                break


    deep_coadd_extent = (deep_coadd.getBBox().beginX,
                     deep_coadd.getBBox().endX,
                     deep_coadd.getBBox().beginY,
                     deep_coadd.getBBox().endY)



    use_columns = ['objectId', 'patch',
               'r_centroid_x', 'r_centroid_y',
               'r_psfFlux', 'r_psfFluxErr']
    
    objects = butler.get('object', tract=dataId.get('tract'),
                     parameters={'columns': use_columns})

    tx = np.where(objects['patch'] == dataId.get('patch'))[0]
    print("Number of objects: ", len(tx))

    sel = objects['r_psfFlux'] / objects['r_psfFluxErr'] > 300
    objects2 = objects[sel]
    tx2 = np.where(objects2['patch'] == dataId.get('patch'))[0]
    print("Number of high SNR objects: ", len(tx2))
    print(f"Stellar ra/dec: {ra} {dec}")
    print(f"in_Obj: {row["in_Obj"]}")
    print(row["DR3_field_loc_center"])

    
    query_make_plot(deep_coadd,row,tx,tx2,objects,objects2,display=dis)

    
  

In [ ]:
query_main()

In [ ]:
# query_main()